# Observed-attrition sample

Combine the 805 completers with the 56 assigned participants who did not finish.

## Load analysed and dropout files

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
DATA = ROOT / "data"
PSY_PATH = DATA / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
DROP_PATH = DATA / "Psy_Case_Vignette_Data_Prolific_dropouts.xlsx"
OUT_PATH = DATA / "Psy_Case_Vignette_Data_Prolific_observed_attrition.xlsx"

VIGNETTES = list("ABCDE")
ARM_LABELS = {1: "ChatGPT", 2: "Copilot", 3: "Perplexity", 4: "Control"}

COND_PRE = [f"vig_{v}_pre_condition_correct" for v in VIGNETTES]
COND_POST = [f"vig_{v}_post_condition_correct" for v in VIGNETTES]
TRIAGE_PRE = [f"vig_{v}_pre_nextstep_correct" for v in VIGNETTES]
TRIAGE_POST = [f"vig_{v}_post_nextstep_correct" for v in VIGNETTES]
SCORE_COLS = COND_PRE + COND_POST + TRIAGE_PRE + TRIAGE_POST

psy = pd.read_excel(PSY_PATH)
drop = pd.read_excel(DROP_PATH)

alloc = drop.loc[drop["dropout_stage"] == "allocated_incomplete_post"].copy()
print(f"Analysed: {len(psy):,}")
print(f"Allocated incomplete dropouts: {len(alloc):,}")
print(f"By arm:\n{alloc['study_arm'].map(ARM_LABELS).value_counts().reindex(ARM_LABELS.values()).to_string()}")

Analysed: 805
Allocated incomplete dropouts: 56
By arm:
study_arm
ChatGPT        7
Copilot       13
Perplexity    27
Control        9


## Align columns and combine

In [2]:
def normalize_prolific_id(pid) -> str:
    return str(pid).split("@")[0].strip().lower()


psy = psy.copy()
alloc = alloc.copy()
psy["sample_group"] = "analysed"
alloc["sample_group"] = "allocated_incomplete"

psy_norm = set(psy["prolific_id"].map(normalize_prolific_id))
alloc_norm = set(alloc["prolific_id"].map(normalize_prolific_id))
overlap = psy_norm & alloc_norm
if overlap:
    raise RuntimeError(f"ID overlap between analysed and dropouts: {len(overlap)}")

# Psy schema columns only (drop meta-only dropout columns not in Psy)
psy_cols = [c for c in psy.columns if c != "sample_group"]
missing_in_alloc = [c for c in psy_cols if c not in alloc.columns]
for col in missing_in_alloc:
    alloc[col] = np.nan

combined = pd.concat(
    [psy[psy_cols + ["sample_group"]], alloc[psy_cols + ["sample_group"]]],
    ignore_index=True,
)

print(f"Combined wide rows: {len(combined):,} ({combined['sample_group'].value_counts().to_dict()})")
print(f"Unique prolific_id: {combined['prolific_id'].nunique():,}")

Combined wide rows: 861 ({'analysed': 805, 'allocated_incomplete': 56})
Unique prolific_id: 861


## QA: observed person–vignette cells

In [3]:
def count_nonnull(df: pd.DataFrame, cols: list[str]) -> int:
    return int(df[cols].notna().sum().sum())


def long_obs_counts(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group, sub in df.groupby("sample_group", observed=True):
        rows.append(
            {
                "sample_group": group,
                "n_participants": sub["prolific_id"].nunique(),
                "pre_condition_cells": count_nonnull(sub, COND_PRE),
                "post_condition_cells": count_nonnull(sub, COND_POST),
                "pre_triage_cells": count_nonnull(sub, TRIAGE_PRE),
                "post_triage_cells": count_nonnull(sub, TRIAGE_POST),
            }
        )
    return pd.DataFrame(rows)


qa = long_obs_counts(combined)
qa.loc[len(qa)] = {
    "sample_group": "TOTAL",
    "n_participants": combined["prolific_id"].nunique(),
    "pre_condition_cells": count_nonnull(combined, COND_PRE),
    "post_condition_cells": count_nonnull(combined, COND_POST),
    "pre_triage_cells": count_nonnull(combined, TRIAGE_PRE),
    "post_triage_cells": count_nonnull(combined, TRIAGE_POST),
}
print(qa.to_string(index=False))

print("\nAllocated incomplete: post cells answered per person")
print(
    combined.loc[combined["sample_group"] == "allocated_incomplete", COND_POST]
    .notna()
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .to_string()
)

# Expected: analysed contribute 805*5 pre and post; incomplete add 56*5 pre and ~102 post condition cells
post_cells_incomplete = count_nonnull(
    combined.loc[combined["sample_group"] == "allocated_incomplete"], COND_POST
)
print(f"\nPost condition cells from incomplete: {post_cells_incomplete} (max possible 280)")

        sample_group  n_participants  pre_condition_cells  post_condition_cells  pre_triage_cells  post_triage_cells
allocated_incomplete              56                  280                   102               280                101
            analysed             805                 4025                  4025              4025               4025
               TOTAL             861                 4305                  4127              4305               4126

Allocated incomplete: post cells answered per person
1    26
2    20
3     6
4     2
5     2

Post condition cells from incomplete: 102 (max possible 280)


## Save combined wide file

In [4]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
combined.to_excel(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}")
print(f"Columns: {combined.shape[1]}; rows: {combined.shape[0]}")

Saved C:\ResearchEnvironment\papers\Prolific\data\Psy_Case_Vignette_Data_Prolific_observed_attrition.xlsx
Columns: 128; rows: 861
